# Multi-Agent RAG System — TechNova Solutions

Este notebook demuestra el sistema multi-agente RAG de TechNova Solutions.
Cubre desde la carga de documentos y vector stores hasta el enrutamiento inteligente
con el orquestador, pasando por la definición de cada agente especializado.

## 1. Setup e Imports

En esta sección configuramos el entorno de trabajo: ajustamos el `sys.path` para que
los módulos del paquete `src` sean importables desde el directorio `notebooks/`,
cargamos las variables de entorno desde `.env` y verificamos que las claves necesarias
estén disponibles. También mostramos las versiones de las librerías clave.

In [ ]:
import os
import sys
import json

# ── Path & working-directory fix ───────────────────────────────────────────────
# Notebooks live in notebooks/, but src/ and data/ are one level up.
# We change cwd to the project root so that relative paths (e.g. data/hr_docs)
# resolve correctly, and insert ".." at the front of sys.path so `from src.x
# import Y` works without installing the package.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, "..")

# Make the project root importable even if the kernel was started elsewhere
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Working directory : {os.getcwd()}")
print(f"sys.path[0]       : {sys.path[0]}")

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads .env from the project root (current working directory)

# ── Verify required API keys ───────────────────────────────────────────────────
openai_key = os.getenv("OPENAI_API_KEY", "")
assert openai_key, "OPENAI_API_KEY is not set — add it to your .env file"
print(f"OPENAI_API_KEY    : {'*' * 8}{openai_key[-4:]}")

langfuse_pub = os.getenv("LANGFUSE_PUBLIC_KEY", "")
langfuse_sec = os.getenv("LANGFUSE_SECRET_KEY", "")
langfuse_host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

if langfuse_pub and langfuse_sec:
    print(f"LANGFUSE_PUBLIC_KEY : set ({langfuse_pub[:8]}...)")
    print(f"LANGFUSE_SECRET_KEY : set")
    print(f"LANGFUSE_HOST       : {langfuse_host}")
else:
    print("LANGFUSE keys not set — tracing will be skipped")

In [ ]:
import importlib.metadata as meta

# ── Library versions ───────────────────────────────────────────────────────────
packages = ["langchain", "langfuse", "chromadb", "openai"]
for pkg in packages:
    try:
        version = meta.version(pkg)
    except meta.PackageNotFoundError:
        version = "not installed"
    print(f"{pkg:<12} : {version}")

In [ ]:
from src.config import Settings

settings = Settings()
print(f"MODEL_NAME   : {settings.MODEL_NAME}")
print(f"CHUNK_SIZE   : {settings.CHUNK_SIZE}")
print(f"CHUNK_OVERLAP: {settings.CHUNK_OVERLAP}")

## 2. Carga de Documentos y Vector Stores

El sistema RAG necesita que los documentos de cada dominio (RRHH, Tecnología y Finanzas)
estén indexados en colecciones vectoriales de ChromaDB.

- `DocumentLoader.load_and_split()` lee los archivos Markdown de cada dominio y los
  fragmenta en *chunks* usando `RecursiveCharacterTextSplitter`.
- `VectorStoreManager.initialize_all_stores()` crea o recarga las tres colecciones en
  `./chroma_db`. Si ya existen en disco, las reutiliza sin re-embedir.

In [ ]:
from src.document_loader import DocumentLoader
from src.vector_store import VectorStoreManager

loader = DocumentLoader()

# ── Load and split each domain ─────────────────────────────────────────────────
domains = {
    "hr"     : "data/hr_docs",
    "tech"   : "data/tech_docs",
    "finance": "data/finance_docs",
}

domain_chunks = {}
for domain, directory in domains.items():
    chunks = loader.load_and_split(directory=directory, domain=domain)
    domain_chunks[domain] = chunks
    print(f"  → {domain:8s}: {len(chunks):4d} chunks from '{directory}'")

print("\nTotal chunks:", sum(len(c) for c in domain_chunks.values()))

In [ ]:
# ── Initialize (or reload) vector stores ──────────────────────────────────────
vsm = VectorStoreManager()
stores = vsm.initialize_all_stores()

print("\n── Collection stats ──")
for collection_name, store in stores.items():
    count = store._collection.count()
    print(f"  {collection_name:<15}: {count:4d} chunks")

In [ ]:
# ── Quick retrieval smoke-test on hr_docs ─────────────────────────────────────
test_query = "What is TechNova's vacation policy?"
hr_retriever = vsm.get_retriever("hr_docs", k=3)
results = hr_retriever.invoke(test_query)

print(f"Query : '{test_query}'")
print(f"Chunks retrieved: {len(results)}\n")

for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source_file", "unknown")
    preview = doc.page_content[:200].replace("\n", " ")
    print(f"[{i}] source: {source}")
    print(f"    {preview}...\n")

## 3. Definición de Agentes RAG

Cada dominio tiene un agente especializado que encapsula un pipeline RAG:

| Agente | Dominio | Retriever |
|--------|---------|----------|
| `HRAgent` | Recursos Humanos | `hr_docs` |
| `TechAgent` | Tecnología / IT | `tech_docs` |
| `FinanceAgent` | Finanzas | `finance_docs` |

Todos heredan de `BaseRAGAgent` y exponen el método `invoke(query) -> dict`
con las claves `answer`, `sources`, `agent_name` y `domain`.

In [ ]:
from src.agents import HRAgent, TechAgent, FinanceAgent
from langchain_openai import ChatOpenAI

# ── Shared LLM ────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model=settings.MODEL_NAME,
    api_key=settings.OPENAI_API_KEY,
)

# ── Retrievers ────────────────────────────────────────────────────────────────
hr_retriever      = vsm.get_retriever("hr_docs")
tech_retriever    = vsm.get_retriever("tech_docs")
finance_retriever = vsm.get_retriever("finance_docs")

# ── Instantiate agents ────────────────────────────────────────────────────────
hr_agent      = HRAgent(hr_retriever, llm)
tech_agent    = TechAgent(tech_retriever, llm)
finance_agent = FinanceAgent(finance_retriever, llm)

print("Agents instantiated:")
for agent in [hr_agent, tech_agent, finance_agent]:
    print(f"  {agent.agent_name} (domain={agent.domain})")

In [ ]:
# ── HR Agent test ─────────────────────────────────────────────────────────────
hr_result = hr_agent.invoke("What are the employee benefits at TechNova?")

print(f"Agent : {hr_result['agent_name']} ({hr_result['domain']})")
print(f"\nAnswer:\n{hr_result['answer']}")
print(f"\nSources ({len(hr_result['sources'])}):", hr_result['sources'][:3])

In [ ]:
# ── Tech Agent test ───────────────────────────────────────────────────────────
tech_result = tech_agent.invoke("How do I set up the VPN?")

print(f"Agent : {tech_result['agent_name']} ({tech_result['domain']})")
print(f"\nAnswer:\n{tech_result['answer']}")
print(f"\nSources ({len(tech_result['sources'])}):", tech_result['sources'][:3])

In [ ]:
# ── Finance Agent test ────────────────────────────────────────────────────────
finance_result = finance_agent.invoke("What is the expense reimbursement process?")

print(f"Agent : {finance_result['agent_name']} ({finance_result['domain']})")
print(f"\nAnswer:\n{finance_result['answer']}")
print(f"\nSources ({len(finance_result['sources'])}):", finance_result['sources'][:3])

## 4. Orquestador y Enrutamiento Inteligente

El `Orchestrator` unifica todo el pipeline:

1. **`classify_intent`** — clasifica la consulta del usuario en uno de los dominios
   (`hr`, `tech`, `finance`, `unknown`) usando un LLM con un prompt estructurado.
2. **`route`** — clasifica y despacha la consulta al agente correcto, creando un
   trace completo en Langfuse con spans para clasificación y recuperación RAG.
3. **`batch_route`** — procesa una lista de consultas y calcula la precisión del
   enrutamiento comparando intenciones predichas contra las esperadas.

La respuesta de `route` incluye: `query`, `intent`, `confidence`, `reasoning`,
`answer`, `sources` y `agent`.

In [ ]:
from src.agents import Orchestrator, classify_intent

# ── Instantiate orchestrator ──────────────────────────────────────────────────
# Orchestrator creates its own Settings, LLM, VectorStoreManager and agents
# internally, so it is self-contained.
orch = Orchestrator()

# ── Demo classify_intent directly ─────────────────────────────────────────────
classification_query = "How do I request time off?"
classification = classify_intent(classification_query, orch.llm)

print(f"Query      : '{classification_query}'")
print(f"Intent     : {classification['intent']}")
print(f"Confidence : {classification['confidence']:.2f}")
print(f"Reasoning  : {classification['reasoning']}")

In [ ]:
# ── Full route demo ───────────────────────────────────────────────────────────
route_query = "How do I request time off?"
response = orch.route(route_query)

print(f"Query      : {response['query']}")
print(f"Intent     : {response['intent']}  (confidence={response['confidence']:.2f})")
print(f"Reasoning  : {response['reasoning']}")
print(f"Agent used : {response['agent']}")
print(f"\nAnswer:\n{response['answer']}")
print(f"\nSources ({len(response['sources'])}):", response['sources'][:3])

In [ ]:
# ── Multi-domain routing demo ─────────────────────────────────────────────────
demo_queries = [
    "How do I configure multi-factor authentication for my work account?",
    "What documentation do I need to submit for international travel expenses?",
]

for query in demo_queries:
    result = orch.route(query)
    print("=" * 70)
    print(f"Query      : {result['query']}")
    print(f"Intent     : {result['intent']}  (confidence={result['confidence']:.2f})")
    print(f"Agent used : {result['agent']}")
    # Truncate answer for readability
    answer_preview = result['answer'][:300].replace("\n", " ")
    print(f"Answer     : {answer_preview}{'...' if len(result['answer']) > 300 else ''}")
    print(f"Sources    : {result['sources'][:2]}")
    print()